In [ ]:
pip install delta-spark==2.4.0

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

spark = (
    SparkSession.builder
    .appName("Feriados ETL")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
    .getOrCreate()
)

In [ ]:
# --- 1. EXTRAÇÃO (Extract) ---

schema_feriados = StructType([
    StructField("date", StringType(), True),      # lida como texto e convertida na limpeza
    StructField("holiday", StringType(), True),
    StructField("weekday", StringType(), True),
    StructField("month", IntegerType(), True),
    StructField("day", IntegerType(), True),
    StructField("year", IntegerType(), True),
])

csv_path = "hdfs://hdfs-nn:9000/projeto/bronze/us_holiday_dates.csv"

df = spark.read.csv(
    csv_path,
    header=True,
    schema=schema_feriados,
    encoding="UTF-8",
    quote='"',
    escape='"'
)

print(f"Total de registos lidos: {df.count()}")      # esperado: 342
df.show(5, truncate=30)

In [ ]:
# --- 2. TRANSFORMAÇÃO (Transform) ---

print("\n=== Início da Limpeza e Transformação ===")

# 2.1. Eliminar weekday, month, day e year (deriváveis da data)
# month e day têm erros em 6 linhas "Labor Day Weekend" (p.ex. 2008-08-30 vem com month=9 e day=30)
erradas = df.filter((F.month(F.to_date("date")) != F.col("month")) |
                    (F.dayofmonth(F.to_date("date")) != F.col("day"))).count()
print(f"Linhas em que month/day não batem com a data: {erradas}")      # esperado: 6
df_clean = df.drop("weekday", "month", "day", "year")
print("TRATAMENTO: Eliminadas 4 colunas (weekday, month, day, year).")

# 2.2. Data (texto -> date) e chave de partição
df_clean = df_clean.withColumn("data", F.to_date("date", "yyyy-MM-dd"))
assert df_clean.filter(F.col("data").isNull()).count() == 0, "Há datas de feriados que não foram interpretadas"
df_clean = df_clean.withColumn("ano", F.year("data"))
print("TRATAMENTO: Datas convertidas para DATE e criada a coluna 'ano' (partição).")

# 2.3. Nome do feriado: espaços a mais e apóstrofo uniforme
# ("New Year’s Eve" e "Valentine’s Day" usam ’, enquanto "New Year's Day" usa ')
df_clean = df_clean.withColumn(
    "feriado",
    F.regexp_replace(F.regexp_replace(F.trim(F.col("holiday")), "’", "'"), r"\s+", " ")
)
assert df_clean.filter(F.col("feriado").isNull() | (F.col("feriado") == "")).count() == 0
print("TRATAMENTO: Nomes limpos (espaços e apóstrofos uniformizados).")

# 2.4. Remover duplicados (mesma data e mesmo feriado)
total_antes = df_clean.count()
df_clean = df_clean.dropDuplicates(["data", "feriado"]).select("data", "feriado", "ano")
print(f"TRATAMENTO: Removidos {total_antes - df_clean.count()} duplicados.")      # esperado: 0

# 2.5. Filtrar o período: só o ano que coincide com as datas dos datasets do projeto
ANO_MIN, ANO_MAX = 2017, 2017
total_antes = df_clean.count()
df_clean = df_clean.filter(F.col("ano").between(ANO_MIN, ANO_MAX))
print(f"TRATAMENTO (Filtro período): Removidos {total_antes - df_clean.count()} registos fora de {ANO_MIN}-{ANO_MAX}.")      # esperado: 323

In [ ]:
# --- 3. CARGA (Load) ---

silver_path = "hdfs://hdfs-nn:9000/projeto/silver/feriados_eua"

spark.sql(f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{warehouse_location}/silver.db'")

print(f"\n=== A guardar dados limpos em {silver_path} ===")

# Guardar em formato Parquet particionado por ano
df_clean.write \
    .mode("overwrite") \
    .format("parquet") \
    .partitionBy("ano") \
    .save(silver_path)

# Criar tabela Hive externa associada ao Parquet
spark.sql("DROP TABLE IF EXISTS silver.feriados_eua")

spark.sql(f"""
    CREATE EXTERNAL TABLE silver.feriados_eua (
        `data` DATE,
        `feriado` STRING
    )
    PARTITIONED BY (`ano` INT)
    STORED AS PARQUET
    LOCATION '{silver_path}'
""")

# Atualizar partições
spark.catalog.recoverPartitions("silver.feriados_eua")

print("\nTabela Parquet particionada criada com sucesso em silver.feriados_eua")

In [ ]:
# --- 4. VERIFICAÇÃO ---

print("\n=== Total de feriados ===")
spark.sql("SELECT COUNT(*) AS total FROM silver.feriados_eua").show()      # esperado: 19

print("\n=== Feriados por ano ===")
spark.sql("""
    SELECT ano, COUNT(*) AS feriados
    FROM silver.feriados_eua
    GROUP BY ano
    ORDER BY ano
""").show(30)      # esperado: 19 em 2017

print("\n=== Ocorrências por feriado ===")
spark.sql("""
    SELECT feriado, COUNT(*) AS ocorrencias
    FROM silver.feriados_eua
    GROUP BY feriado
    ORDER BY feriado
""").show(30, truncate=False)      # esperado: 1 por feriado (Labor Day Weekend: 2)

spark.stop()

In [ ]:
from pyspark.sql import SparkSession

# ATENÇÃO: esta célula tem de correr num kernel novo (Kernel > Restart Kernel), para o Spark carregar o Delta.
# Se a conversão der PARSE_SYNTAX_ERROR em 'CONVERT', reinicia o kernel e corre só as 3 últimas células.

# --- 1. CONFIGURAÇÃO DA SPARKSESSION (com Delta) ---
# Esta célula precisa da sua própria SparkSession, que carrega as bibliotecas do Delta.
print("A iniciar a SparkSession com o pacote Delta Lake...")

warehouse_location = "hdfs://hdfs-nn:9000/warehouse"
hive_uris = "thrift://hive-metastore:9083"

spark = (
    SparkSession.builder
    .appName("Delta Converter (Feriados)")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", hive_uris)

    # --- Configuração essencial do Delta ---
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    .enableHiveSupport()
    .getOrCreate()
)

print("Sessão Spark com Delta pronta.")

In [ ]:
# --- 2. O COMANDO DE CONVERSÃO ---
# Não re-escreve os dados Parquet: apenas acrescenta o '_delta_log' para "adotar" os ficheiros existentes.
# Como as tabelas estão particionadas, é obrigatório indicar a coluna de partição.
tabelas = {
    "silver.feriados_eua": "ano"
}

for tabela, particao in tabelas.items():
    print(f"A converter a tabela Parquet '{tabela}' para Delta Lake...")
    try:
        spark.sql(f"""
            CONVERT TO DELTA {tabela}
            PARTITIONED BY ({particao} INT)
        """)
        print(" Conversão concluída com sucesso!")
    except Exception as e:
        print(" Ocorreu um erro durante a conversão.")
        print("Causas prováveis: 1) A tabela não existe. 2) A tabela já é Delta.")
        print(e)

In [ ]:
# --- 3. VERIFICAÇÃO ---
spark.sql("DESCRIBE DETAIL silver.feriados_eua").select("format", "location", "numFiles", "partitionColumns").show(truncate=False)
spark.sql("SELECT COUNT(*) AS feriados FROM silver.feriados_eua").show()      # esperado: 19

# (Opcional) Podes parar esta sessão
# spark.stop()

In [ ]:
# --- 4. TABELA FINAL DA SILVER ---
df_final = spark.table("silver.feriados_eua").orderBy("data")
print("Linhas:", df_final.count(), "| Colunas:", len(df_final.columns))
df_final.printSchema()
df_final.show(df_final.count(), truncate=False)
